In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

In [ ]:
session = get_active_session()

In [ ]:
df=session.table('test.diamonds.diamonds')

In [ ]:
_,test_df=df.random_split(weights=[0.9,0.1],seed=42)

In [ ]:
test_df.show()

In [ ]:
registry=Registry(session=session, 
                  database_name='test',
                  schema_name='diamonds')

In [ ]:
lastModelName=pd.DataFrame(data=registry.show_models()).\
    sort_values('created_on',ascending=False).\
    loc[:1,'name'].values[0]
lastModelName

In [ ]:
model=registry.get_model(lastModelName)
model

In [ ]:
mv=model.default
mv

In [ ]:
dir(mv)

In [ ]:
mv.show_functions()

In [ ]:
mv.create_service(
    service_name="Diamonds_Regression",
    service_compute_pool='SYSTEM_COMPUTE_POOL_GPU'
                 )

In [ ]:
pred=mv.run(test_df)

In [ ]:
-- WITH test_df AS (
--     SELECT  
--         *
--     FROM 
--         test.diamonds.diamonds
--     LIMIT 10  
-- ),
-- preds AS (
--     WITH v1 AS 
--         model test.public.{{lastModelName}} VERSION V2
--         SELECT
--             price,v2!predict(
--                 DEPTH,
--                 TABLENO,
--                 x,
--                 y,
--                 z  
--             ) AS pred
--         FROM 
--             test_df)
-- SELECT 
--     price, 
--     pred:predicted_price
-- FROM    
--     preds;


In [ ]:
WITH 
test_df AS 
(
    SELECT 
        *
    FROM    
        test.diamonds.diamonds
    LIMIT 
        10
),
PREDS AS 
(
    SELECT 
        price,
        test.diamonds.{{lastModelName}}!predict(
                DEPTH,
                TABLENO,
                x,
                y,
                z  
            ) AS PRED
FROM test_df)
SELECT 
    price,
    pred:PREDICTED_PRICE
FROM 
    preds;